# 02/SageMaker SFT 학습 (TRL/LoRA): 문서 요약

**요약**: `scripts/train.py`를 SageMaker ModelTrainer로 실행합니다(SFT). 로컬 dry-run에서 쓰는 스크립트와 완전히 동일합니다. (GRPO 등 다른 학습법은 별도 노트북.)

**목적**: 학습 스크립트가 self-contained이므로, 로컬 GPU에서 dry-run으로 먼저 검증한 뒤 같은 스크립트를 그대로 클라우드 학습 잡으로 제출할 수 있습니다.

**배경**: 학습 코드를 클라우드용과 로컬용으로 나눠 관리하면 두 버전이 서로 어긋나기 쉽습니다. 하나의 파일로 통일해 이러한 drift를 원천적으로 막습니다.

> 실제 실행에는 AWS 자격증명과 비용이 필요합니다. 먼저 `DRY_RUN=1`로 파이프라인을 검증하세요.

In [ ]:
import os, sys
# 리포 루트를 path에 추가해 common/ 와 트랙 로컬 모듈을 import
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, REPO)
sys.path.insert(0, os.getcwd())

In [ ]:
import importlib, boto3, os
from common import config, aws_utils; importlib.reload(config)
from sagemaker.core.helper.session_helper import Session
sess = Session(boto3.Session(region_name=config.AWS_REGION))
%store -r role
%store -r bucket
# train_path는 %store(트랙 공유) 대신 이 트랙의 로컬 파일로 고정: 트랙 간 값 오염 방지.
train_path = 'data/train.jsonl'
assert os.path.isfile(train_path), (
    f'{train_path} 가 없습니다. 이 트랙의 01_data_and_synthetic.ipynb 를 먼저 실행해 '
    '학습 데이터를 생성하세요. (%store는 트랙 간 공유되므로 다른 트랙 경로를 쓰면 안 됩니다.)')
# %store 오염 방지: role이 없거나 옛 플레이스홀더면 다시 해석.
if 'role' not in dir() or not role or ':role/' not in str(role):
    role = config.resolve_sagemaker_role(sess)
if 'bucket' not in dir() or not bucket:
    bucket = config.S3_BUCKET or sess.default_bucket()
print('train_path:', train_path, f'({sum(1 for _ in open(train_path))} lines)')
print('role      :', role)

## 1. `scripts/train.py` 학습 로직 이해하기
이 트랙의 학습은 `scripts/train.py` 한 파일이 담당하며, **로컬 dry-run과 SageMaker 학습 잡에서 동일하게** 실행됩니다.
SageMaker로 넘기기 전에 스크립트가 수행하는 핵심 단계를 짚어봅니다.

1. **데이터 로드**: `messages` 컬럼(conversational)을 가진 JSONL을 `datasets`로 로드합니다.
2. **chat template 자동 적용**: TRL `SFTTrainer`는 conversational 데이터셋을 받으면 토크나이저의
   `apply_chat_template`을 자동 호출합니다. 즉 `<start_of_turn>` 마커를 직접 조립하지 않습니다.
3. **LoRA 설정**: `target_modules='all-linear'` + `modules_to_save=['lm_head','embed_tokens']`.
   Gemma의 특수 토큰 임베딩까지 학습하기 위해 두 모듈을 저장 대상에 포함합니다.
4. **정밀도**: `bf16=True`. Gemma는 fp16에서 오버플로/NaN이 발생하므로 fp16을 쓰지 않습니다.
5. **packing 안전장치**: `attn_implementation`이 flash-attention이 아니면(기본 `eager`) packing을
   비활성화합니다. packing은 여러 샘플을 한 시퀀스로 합치는데, flash-attention이 아닐 경우 샘플 간
   cross-contamination 위험이 있기 때문입니다.
6. **QLoRA(선택)**: `--use_qlora True`이면 4-bit(nf4, double-quant)로 로드해 단일 GPU 메모리를 절약합니다.
7. **어댑터 저장 & 머지(선택)**: 학습된 LoRA 어댑터를 저장하고, `--merge_adapter True`이면 base에 병합해
   서빙용 단일 모델을 만듭니다.

핵심 스니펫(설명용 발췌: 실제 실행은 아래 dry-run 셀 및 `scripts/train.py` 전체):
```python
peft_config = LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules='all-linear',                  # Gemma 관용구
    modules_to_save=['lm_head', 'embed_tokens'],  # 특수 토큰 임베딩까지 학습
)
sft_config = SFTConfig(
    bf16=True,                                    # fp16 금지 (Gemma NaN)
    packing=use_packing,                          # flash-attn 아니면 자동 off
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    max_length=max_seq_length, optim='adamw_torch_fused',
)
# SFTTrainer가 conversational 'messages' 데이터에 chat template을 자동 적용
trainer = SFTTrainer(model, args=sft_config, train_dataset=ds,
                     peft_config=peft_config, processing_class=tokenizer)
```

### (선택) 로컬 GPU dry-run으로 먼저 검증 (권장)
클라우드 학습 잡은 인스턴스 시간만큼 과금되므로, GPU 개발환경이 있다면 클라우드로 제출하기 전에
로컬에서 스크립트를 짧게 실행해 파이프라인(데이터 로드 → 토크나이즈 → 몇 step 학습 → 저장)이
정상 동작하는지 확인합니다. `--dry_run` 플래그가 epochs=1, 짧은 시퀀스, 최대 32행으로 축소하여
수 분 내에 끝납니다.

In [ ]:
!cd scripts && python train.py --dry_run \
    --model_id {config.DEFAULT_MODEL_ID} \
    --train_file ../{train_path} \
    --max_seq_length 2048 \
    --output_dir ../out_dryrun

## 2. 학습 데이터 S3 업로드
SageMaker 학습 잡은 격리된 컨테이너에서 실행되며 로컬 파일 시스템에 접근할 수 없으므로, 학습 데이터를 먼저 S3에 업로드해야 합니다. 학습 시작 시 이 S3 경로가 학습 컨테이너의 입력 채널로 마운트됩니다.

In [ ]:
import os
# 내용 해시 비교로 조건부 업로드: 여러 번 실행해도 바뀐 게 없으면 재업로드 안 함(빠름/무료).
key = f'{config.S3_PREFIX}/summarization/data/' + os.path.basename(train_path)
train_s3 = aws_utils.upload_if_changed(train_path, bucket, key, region=config.AWS_REGION)

## 3. ModelTrainer 구성 (JumpStart 아님: DLC + 커스텀 TRL 스크립트)
sagemaker SDK v3에서는 학습을 `ModelTrainer`로 정의합니다(v2의 `HuggingFace`/`Estimator`는 제거됨).
우리의 `scripts/train.py`는 그대로 두고, `SourceCode`로 스크립트를, `Compute`로 인스턴스를, `training_image`로
DLC 이미지를 지정합니다.
- **DLC 이미지**는 `.env`의 `DLC_IMAGE_URI`(리전 포함 완전 URI)로 하드코딩해 뒀습니다 -
  `common/dlc.resolve_training_image()`가 그 값을 그대로 씁니다. 무엇이 쓰이는지 한눈에 보이고,
  SDK가 아는 버전 목록에 매이지 않습니다(그 목록은 최신 태그를 모를 수 있습니다).
  `train.py`가 `scripts/requirements.txt`로 필요한 라이브러리를 직접 설치하므로, 베이스가 순수
  PyTorch DLC여도 최신 transformers/trl/peft를 컨테이너 안에서 맞출 수 있어 유리합니다.
- 주의: 리전을 옮길 땐 `AWS_REGION`과 `DLC_IMAGE_URI`의 리전을 함께 바꾸세요(이미지는 리전별 ECR에서만 pull).
  현행 태그 확인: `aws ecr describe-images --registry-id 763104351884 --repository-name pytorch-training --region <region>`

**핸즈온 기본값은 짧게 잡았습니다**: `MAX_TRAIN_SAMPLES=200`, `EPOCHS=2`. 실습에서 파이프라인이 끝까지 도는지 확인하는 것이 목적이므로, 전량/다epoch 학습은 `MAX_TRAIN_SAMPLES=None` / `EPOCHS=3~5`로 올려서 따로 돌리세요.
- `stopping_condition`도 **반드시 명시**합니다: 생략 시 SDK 기본값 1시간에 걸려 학습이 끝난 뒤 머지 단계에서 잡이 죽습니다(바로 아래 절에 실측).

In [ ]:
from common import dlc
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.core.training.configs import SourceCode, Compute, InputData, StoppingCondition

# ── 실습 규모 (시간/비용 조절) ──
MAX_TRAIN_SAMPLES = 200   # train.jsonl 앞 N건만 학습(파일은 그대로). 정식 학습은 None(전체).
EPOCHS = 2                # 실습 2 / 정식 3~5
MAX_RUNTIME_HOURS = 4     # 초과 시 강제 중단(아래 절 참고)

hyperparameters = {
    'model_id': config.DEFAULT_MODEL_ID,
    'epochs': EPOCHS, 'per_device_train_batch_size': 1, 'gradient_accumulation_steps': 8,
    'learning_rate': 2e-4,
    'max_seq_length': 2048,
    'lora_r': 16, 'lora_alpha': 16, 'lora_dropout': 0.05,
    'use_qlora': True, 'merge_adapter': True,
}
if MAX_TRAIN_SAMPLES:
    hyperparameters['max_train_samples'] = MAX_TRAIN_SAMPLES

# step = ceil(건수/8) x epochs. 실측 g6.2xlarge: seq2048≈17s, seq512≈7s/step
_n = MAX_TRAIN_SAMPLES or sum(1 for _ in open(train_path))
_steps = -(-_n // 8) * EPOCHS
_eta = _steps * 17 / 60
print(f'학습 {_n}건 x {EPOCHS}epoch = 약 {_steps} step '
      f'-> 학습 ~{_eta:.0f}분 + 머지/업로드 ~5분 (한도 {MAX_RUNTIME_HOURS}시간)')
assert _eta / 60 < MAX_RUNTIME_HOURS, (
    f'예상 학습 시간({_eta:.0f}분)이 MAX_RUNTIME_HOURS({MAX_RUNTIME_HOURS}시간)에 육박합니다. '
    'MAX_TRAIN_SAMPLES/EPOCHS를 줄이거나 MAX_RUNTIME_HOURS를 올리세요.')
environment = {'HF_TOKEN': config.get_hf_token()} if config.get_hf_token() else {}

# 학습용 DLC 이미지: .env의 DLC_IMAGE_URI(완전 URI)를 그대로 사용.
image_uri = dlc.resolve_training_image(config.AWS_REGION)
assert image_uri, (
    '학습 이미지 해석 실패: .env의 DLC_IMAGE_URI를 확인하세요(리전 포함 완전 URI). '
    '태그 목록: ' + dlc.AVAILABLE_IMAGES_URL)
print('DLC training image:', image_uri)

trainer = ModelTrainer(
    training_image=image_uri,
    source_code=SourceCode(source_dir='scripts', entry_script='train.py',
                           requirements='requirements.txt'),
    compute=Compute(instance_type=config.TRAIN_INSTANCE_TYPE, instance_count=1),
    hyperparameters=hyperparameters,
    environment=environment,
    role=role,
    sagemaker_session=sess,
    base_job_name='gemma-summarization-train',
    # 반드시 명시: 생략 시 SDK 기본 1시간(아래 절 참고)
    stopping_condition=StoppingCondition(max_runtime_in_seconds=MAX_RUNTIME_HOURS * 3600),
)

### `MaxRuntimeExceeded`: 학습이 끝났는데 잡이 `Stopped`로 죽는 함정
`stopping_condition`을 **생략하면 SDK가 `max_runtime_in_seconds=3600`(1시간)을 자동으로 넣습니다** (`sagemaker/train/defaults.py`). 이 값은 학습 코드 시간만이 아니라 **Pending(용량 대기) + Downloading(이미지 pull) + Training + 머지/업로드 전체**를 포함하므로, 실제 학습에 쓸 수 있는 시간은 1시간보다 짧습니다.

실측(03_summarization, `gemma-summarization-train-20260731084146`, ml.g6.2xlarge):

| 단계 | 시간 |
|---|---|
| Pending (용량 대기) | 6분 |
| Downloading (이미지 pull) | 3분 |
| Training: 189 step **전부 완료** | 55분 |
| ⛔ 머지 도중 강제 종료 | 1시간 도달 |

**학습은 100% 끝났는데도 결과물이 버려졌습니다.** LoRA 어댑터를 base에 머지하는 마지막 단계(실측 ~2분)에서 잘려, 아티팩트에 `adapter/`와 `checkpoint-*/`만 남고 **서빙용 머지 모델이 없어** 배포가 불가능했습니다. `MaxRuntimeExceeded`는 `FailureReason`도 비어 있어(상태만 `Stopped`) 원인을 찾기 어렵습니다.

그래서 이 노트북은 `MAX_RUNTIME_HOURS`(기본 4시간)를 **명시**합니다. 넉넉히 잡아도 손해가 없습니다: 잡이 정상 종료되면 그 시점에 과금이 멈추므로, 이 값은 요금이 아니라 **폭주 방지 상한**입니다.
> 여유가 필요하면 `MAX_RUNTIME_HOURS`만 올리세요(API 최대 28일). 반대로 실습 비용을 확실히 막고 싶으면 낮추되, 머지/업로드용으로 **최소 15분은 남겨** 두세요.

## 4. 학습 시작 (.train, 비동기 제출) + CloudWatch 링크
`train(wait=False)`로 학습 잡을 **비동기 제출**합니다. 이렇게 하면 셀이 잡 완료까지 블로킹하지 않고 바로 반환되므로, 아래에서 출력하는 CloudWatch/콘솔 링크로 진행 상황과 로그를 실시간 확인할 수 있습니다.
(`wait=True`로 두면 잡이 끝날 때까지 'Waiting for TrainingJob...' 패널이 계속 갱신되어 링크를 그동안 볼 수 없습니다.)
학습 데이터는 `InputData`로 `train` 채널에 연결되어 컨테이너의 `SM_CHANNEL_TRAIN` 경로로 마운트됩니다.

In [ ]:
trainer.train(input_data_config=[InputData(channel_name='train', data_source=train_s3)],
              wait=False, logs=False)   # 비동기 제출: 블로킹 안 함
from IPython.display import display
job = trainer._latest_training_job
print('training job:', job.training_job_name)
display(aws_utils.cw_links(config.AWS_REGION, training_job=job.training_job_name))

### 진행 상태 확인 (이 셀만 반복 실행)
이 셀을 필요할 때마다 다시 실행해 잡의 진행 단계를 봅니다. `Starting → Pending(용량 대기) → Downloading(이미지 pull) → Training(코드 실행)` 순으로 진행되며, **Training 단계부터 CloudWatch 로그가 생깁니다**.

In [ ]:
aws_utils.training_job_status(job.training_job_name, config.AWS_REGION)

### 세션이 끊겼을 때 잡에 다시 붙기 (재접속)
`train(wait=False)`로 제출한 학습 잡은 **SageMaker 서버에서 실행되므로, 노트북 커널이나 VS Code 세션이 끊겨도 잡은 계속 진행됩니다.** 다시 붙으려면 `trainer` 객체를 복구할 필요 없이 **잡 이름으로 조회**하면 됩니다 (v3에서는 `sagemaker.core.resources.TrainingJob.get(name)`). 아래 셀은 커널을 재시작한 뒤 이 노트북 위쪽 설정 셀들(§0~§1)만 실행한 상태에서 바로 쓸 수 있습니다.
> 팁: 위 §4에서 출력된 잡 이름을 메모해 두면 `TrainingJob.get('<잡 이름>')`으로 어느 커널/머신에서도 정확히 그 잡에 붙습니다.

In [ ]:
from sagemaker.core.resources import TrainingJob
# 방법 A: 잡 이름을 알면 바로 붙기 (가장 확실: 위 §4 출력에서 복사)
# job = TrainingJob.get('<여기에 잡 이름>')
# 방법 B: 이름을 잊었으면 base_job_name으로 최근 잡을 찾기 (get_all은 최신순)
jobs = list(TrainingJob.get_all(name_contains='gemma-summarization-train'))
assert jobs, '이 base_job_name으로 제출된 잡이 없습니다. §4를 먼저 실행하세요.'
job = TrainingJob.get(jobs[0].get_name())
job.refresh()
print('reattached to:', job.training_job_name)
print('status       :', job.training_job_status, '/', job.secondary_status)
# 로그를 다시 스트리밍하며 대기하려면(잡이 InProgress일 때). Ctrl-C로 빠져나와도 잡은 계속 돕니다.
# if job.training_job_status == 'InProgress':
#     job.wait(logs=True)
from IPython.display import display
display(aws_utils.cw_links(config.AWS_REGION, training_job=job.training_job_name))

## 5. (선택) 학습 완료 대기 → 모델 아티팩트
잡이 끝나야 모델 아티팩트(S3)가 생깁니다. 아래 셀은 완료될 때까지 상태를 폴링하며 기다립니다. 지금 기다리기 싫으면 이 셀은 건너뛰고, CloudWatch에서 `Completed`를 확인한 뒤 다시 실행해도 됩니다.

In [ ]:
import time
# `job`은 §4의 train 셀 또는 위 '세션이 끊겼을 때' 셀에서 정의됩니다(trainer 객체에 의존하지 않음).
assert 'job' in dir() and job is not None, (
    "job이 없습니다: §4의 train 셀이나 위 '세션이 끊겼을 때 잡에 다시 붙기' 셀을 먼저 실행하세요.")
while True:
    job.refresh()
    st = job.training_job_status
    print('status:', st)
    if st in ('Completed', 'Failed', 'Stopped'):
        break
    time.sleep(30)
assert st == 'Completed', f'학습 잡이 {st} 상태입니다. CloudWatch 로그를 확인하세요.'
# v3 모델 아티팩트 S3 URI = model_artifacts.s3_model_artifacts (v2의 estimator.model_data 대응)
model_data = job.model_artifacts.s3_model_artifacts
print('Training complete. Model artifact:', model_data)
md_summarization = model_data
%store model_data
%store md_summarization

학습이 끝나고 `model_data`가 저장되면, (선택) **02b_local_serve.ipynb**로 배포 전 로컬 서빙 검증을 하거나 (로컬에서도 클라우드와 같은 vLLM으로 확인합니다) 바로 **03_deploy_endpoint.ipynb**로 넘어갑니다.